In [1]:
!pip -q install --no-cache-dir "numpy<2" "pandas"

!pip -q install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu121

!pip -q install pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv \
  -f https://data.pyg.org/whl/torch-2.2.2+cu121.html

!pip -q install torch-geometric

In [2]:
!pip -q uninstall -y numpy pandas
!pip -q install --no-cache-dir "numpy<2" "pandas>=2.2,<2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 266.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 173.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 230.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
pytensor 2.37.0 requires numpy>=2.0, but you hav

In [3]:
import numpy as np, torch
import pandas as pd
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("pandas:", pd.__version__)

numpy: 1.26.4
torch: 2.2.2+cu121
pandas: 2.2.3


In [4]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import TransformerConv
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
from sklearn.metrics import average_precision_score, roc_auc_score
from torch_geometric.loader import NeighborLoader

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
base_path = "/content/drive/MyDrive/dataset_cleaned/LI-Small_Trans.csv"  # change if needed

df = pd.read_csv(base_path)
print("Shape:", df.shape)
df.head()

Shape: (6924041, 21)


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,...,Log Amount Received,_ts,tx_hour,tx_dow,tx_month,tx_day,tx_date,tx_is_weekend,tx_hour_sin,tx_hour_cos
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,...,14.977224,2022-09-01 00:08:00,0,3,9,1,2022-09-01,0,0.0,1.0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,...,7.528310,2022-09-01 00:21:00,0,3,9,1,2022-09-01,0,0.0,1.0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,...,13.292228,2022-09-01 00:00:00,0,3,9,1,2022-09-01,0,0.0,1.0
3,2022/09/01 00:16,3814,8006AD080,3814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,...,2.589267,2022-09-01 00:16:00,0,3,9,1,2022-09-01,0,0.0,1.0
4,2022/09/01 00:00,20,8006AD530,20,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,...,7.987035,2022-09-01 00:00:00,0,3,9,1,2022-09-01,0,0.0,1.0


In [7]:
required_cols = [
    "From Bank", "Account", "To Bank", "Account.1",
    "Is Laundering",
    "Log Amount Received", "_ts",
    "tx_dow", "tx_is_weekend",
    "tx_hour_sin", "tx_hour_cos"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

In [8]:
# Build nodes
src_key = df["From Bank"].astype(str) + "|" + df["Account"].astype(str)
dst_key = df["To Bank"].astype(str) + "|" + df["Account.1"].astype(str)

all_keys = pd.concat([src_key, dst_key], ignore_index=True)
codes, uniques = pd.factorize(all_keys, sort=False)

E = len(src_key)
src = codes[:E].astype("int64")
dst = codes[E:].astype("int64")
num_nodes = len(uniques)

edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)

print("num_nodes:", num_nodes, "num_edges:", E)

num_nodes: 705907 num_edges: 6924041


In [9]:

# Build edge_attr from new features
ts = pd.to_datetime(df["_ts"], errors="coerce")

if ts.isna().any():
    ts = ts.fillna(ts.dropna().min())

t0 = ts.min()
t_sec = (ts - t0).dt.total_seconds().astype(np.float32).to_numpy()

t_max = float(t_sec.max()) if float(t_sec.max()) > 0 else 1.0
t_sec = (t_sec / t_max).astype(np.float32)

dow = pd.to_numeric(df["tx_dow"], errors="coerce").fillna(0).astype(np.int64).to_numpy()
dow = np.clip(dow, 0, 6)
dow_oh = np.eye(7, dtype=np.float32)[dow]
is_weekend = pd.to_numeric(df["tx_is_weekend"], errors="coerce").fillna(0).astype(np.float32).to_numpy()
hour_sin = pd.to_numeric(df["tx_hour_sin"], errors="coerce").fillna(0).astype(np.float32).to_numpy()
hour_cos = pd.to_numeric(df["tx_hour_cos"], errors="coerce").fillna(0).astype(np.float32).to_numpy()
amt = pd.to_numeric(df["Log Amount Received"], errors="coerce").fillna(0).astype(np.float32).to_numpy()

edge_attr = np.column_stack([
    amt,
    t_sec,
    hour_sin,
    hour_cos,
    dow_oh,
    is_weekend
]).astype("float32")

y_edge = pd.to_numeric(df["Is Laundering"], errors="coerce").fillna(0).astype("int64").to_numpy()

print("num_nodes:", num_nodes, "num_edges:", E)

num_nodes: 705907 num_edges: 6924041


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
edge_attr_t = torch.tensor(edge_attr, dtype=torch.float32)
y_edge_t    = torch.tensor(y_edge, dtype=torch.long)

data = Data(num_nodes=num_nodes, edge_index=edge_index, edge_attr=edge_attr_t)

edge_ids = torch.arange(edge_index.size(1), dtype=torch.long)

perm = torch.randperm(edge_ids.numel())
n_train = int(0.80 * perm.numel())
train_eids = edge_ids[perm[:n_train]]
val_eids   = edge_ids[perm[n_train:]]

train_loader = LinkNeighborLoader(
    data,
    edge_label_index=edge_index[:, train_eids],
    edge_label=y_edge_t[train_eids],
    num_neighbors=[15, 10],
    batch_size=4096,
    shuffle=True
)

val_loader = LinkNeighborLoader(
    data,
    edge_label_index=edge_index[:, val_eids],
    edge_label=y_edge_t[val_eids],
    num_neighbors=[15, 10],
    batch_size=4096,
    shuffle=False
)

print("train batches:", len(train_loader), "val batches:", len(val_loader))

EDGE_DIM = data.edge_attr.size(1)
print("EDGE_DIM:", EDGE_DIM)

device: cuda
train batches: 1353 val batches: 339
EDGE_DIM: 12


In [11]:
class EdgeAwareGNN(nn.Module):
    def __init__(self, num_nodes, hidden=128, edge_dim=EDGE_DIM):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, hidden)
        self.conv1 = TransformerConv(hidden, hidden, edge_dim=edge_dim)
        self.conv2 = TransformerConv(hidden, hidden, edge_dim=edge_dim)
        self.mlp = nn.Sequential(
            nn.Linear(hidden * 4, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )

    def forward(self, batch):
        x = self.emb(batch.n_id)
        x = F.relu(self.conv1(x, batch.edge_index, batch.edge_attr))
        x = self.conv2(x, batch.edge_index, batch.edge_attr)

        src_i = batch.edge_label_index[0]
        dst_i = batch.edge_label_index[1]
        hs = x[src_i]
        hd = x[dst_i]
        h = torch.cat([hs, hd, hs * hd, (hs - hd).abs()], dim=-1)
        return self.mlp(h)

model = EdgeAwareGNN(num_nodes=num_nodes, hidden=128, edge_dim=EDGE_DIM).to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
crit = nn.CrossEntropyLoss()

In [12]:
def run_epoch_train(model, loader, opt, crit, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in loader:
        batch = batch.to(device)

        logits = model(batch)
        y = batch.edge_label.to(device)

        loss = crit(logits, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        bs = y.numel()
        total_loss += float(loss.item()) * bs

        pred = logits.argmax(dim=1)
        correct += int((pred == y).sum().item())
        total += int(bs)

    avg_loss = total_loss / max(total, 1)
    acc = correct / max(total, 1)
    return avg_loss, acc


@torch.no_grad()
def run_epoch_eval(model, loader, crit, device, max_batches=None):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    probs_all = []
    ys_all = []

    for i, batch in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break

        batch = batch.to(device)
        logits = model(batch)
        y = batch.edge_label.to(device)

        loss = crit(logits, y)

        bs = y.numel()
        total_loss += float(loss.item()) * bs

        pred = logits.argmax(dim=1)
        correct += int((pred == y).sum().item())
        total += int(bs)

        prob1 = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
        probs_all.append(prob1)
        ys_all.append(y.detach().cpu().numpy())

    avg_loss = total_loss / max(total, 1)
    acc = correct / max(total, 1)

    ys = np.concatenate(ys_all) if ys_all else np.array([])
    ps = np.concatenate(probs_all) if probs_all else np.array([])

    if ys.size > 0 and len(np.unique(ys)) > 1:
        pr_auc = average_precision_score(ys, ps)
        roc_auc = roc_auc_score(ys, ps)
    else:
        pr_auc = np.nan
        roc_auc = np.nan

    pos_rate = float(ys.mean()) if ys.size > 0 else np.nan
    return avg_loss, acc, pos_rate, pr_auc, roc_auc

In [13]:
EPOCHS = 3

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch_train(model, train_loader, opt, crit, device)
    va_loss, va_acc, _, _, _ = run_epoch_eval(model, val_loader, crit, device, max_batches=200)

    print(
        f"epoch {epoch} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
        f"val loss {va_loss:.4f} acc {va_acc:.4f}"
    )

epoch 1 | train loss 0.0052 acc 0.9994 | val loss 0.0039 acc 0.9995
epoch 2 | train loss 0.0035 acc 0.9995 | val loss 0.0035 acc 0.9995
epoch 3 | train loss 0.0026 acc 0.9995 | val loss 0.0043 acc 0.9995


In [14]:
_, _, pos_rate, pr_auc, roc_auc = run_epoch_eval(model, val_loader, crit, device, max_batches=200)

print()
print("Pos rate:", pos_rate)
print("PR-AUC:", pr_auc)
print("ROC-AUC:", roc_auc)


Pos rate: 0.00049560546875
PR-AUC: 0.1341677482105054
ROC-AUC: 0.8254613709113527


# CatBoost

In [15]:
try:
    from catboost import CatBoostClassifier
except ImportError:
    !pip -q install catboost
    from catboost import CatBoostClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 25.5 MB/s eta 0:00:00


In [16]:
def encode_batch(model, batch):
    # returns node embeddings for the sampled nodes in this batch
    x = model.emb(batch.n_id)
    x = F.relu(model.conv1(x, batch.edge_index, batch.edge_attr))
    x = model.conv2(x, batch.edge_index, batch.edge_attr)
    return x

hidden = 128  # MUST match your model hidden size

# in-memory embeddings: (num_nodes, hidden)
H = np.zeros((num_nodes, hidden), dtype=np.float32)

infer_loader = NeighborLoader(
    data,
    input_nodes=torch.arange(num_nodes),
    num_neighbors=[10, 5],
    batch_size=65536,
    shuffle=False,
    num_workers=2,
    persistent_workers=True
)

model.eval()
with torch.no_grad():
    for batch in infer_loader:
        batch = batch.to(device)
        z = encode_batch(model, batch).detach().cpu().numpy()
        H[batch.n_id.cpu().numpy()] = z

print("Embeddings stored in memory:", H.shape, "dtype:", H.dtype)

Embeddings stored in memory: (705907, 128) dtype: float32


In [17]:
def build_catboost_Xy(edge_ids, edge_index, edge_attr_t, y_edge_t, H):
    if torch.is_tensor(edge_ids):
        edge_ids_np = edge_ids.detach().cpu().numpy()
    else:
        edge_ids_np = edge_ids

    src = edge_index[0, edge_ids_np].detach().cpu().numpy()
    dst = edge_index[1, edge_ids_np].detach().cpu().numpy()

    hs = H[src]
    hd = H[dst]
    diff = np.abs(hs - hd)
    had = hs * hd

    eattr = edge_attr_t[edge_ids_np].detach().cpu().numpy()
    X = np.concatenate([hs, hd, diff, had, eattr], axis=1)

    y = y_edge_t[edge_ids_np].detach().cpu().numpy()
    return X, y

sample_eids = torch.arange(0, min(10000, edge_index.size(1)))
X_tmp, y_tmp = build_catboost_Xy(sample_eids, edge_index, data.edge_attr, y_edge_t, H)
print("CatBoost X shape:", X_tmp.shape, "y pos rate:", float(y_tmp.mean()))

CatBoost X shape: (10000, 524) y pos rate: 0.0


In [18]:
def build_catboost_Xy(edge_ids, edge_index, edge_attr_t, y_edge_t, H):
    if hasattr(edge_ids, "detach"):
        edge_ids_np = edge_ids.detach().cpu().numpy()
    else:
        edge_ids_np = np.asarray(edge_ids)

    src = edge_index[0, edge_ids_np].detach().cpu().numpy()
    dst = edge_index[1, edge_ids_np].detach().cpu().numpy()

    hs = H[src]
    hd = H[dst]
    diff = np.abs(hs - hd)
    had = hs * hd

    eattr = edge_attr_t[edge_ids_np].detach().cpu().numpy()
    X = np.concatenate([hs, hd, diff, had, eattr], axis=1)

    y = y_edge_t[edge_ids_np].detach().cpu().numpy()
    return X, y


In [19]:
ts = pd.to_datetime(df["_ts"], errors="coerce")
if ts.isna().any():
    ts = ts.fillna(ts.dropna().min())

# sort edges by timestamp
order = np.argsort(ts.values.astype("datetime64[ns]"))
E = len(order)
cut = int(0.80 * E)

train_eids = order[:cut]
test_eids  = order[cut:]

y_np = y_edge_t.detach().cpu().numpy()
train_pos = train_eids[y_np[train_eids] == 1]
train_neg = train_eids[y_np[train_eids] == 0]

test_pos_rate = float(y_np[test_eids].mean())
train_pos_rate = float(y_np[train_eids].mean())

print("Train pos rate:", train_pos_rate)
print("Test pos rate:", test_pos_rate)
print("Train edges:", len(train_eids), "Test edges:", len(test_eids))

Train pos rate: 0.00047660036626016026
Test pos rate: 0.0006679621521812755
Train edges: 5539232 Test edges: 1384809


In [20]:
neg_mult = 10
n_pos = len(train_pos)

if n_pos == 0:
    raise ValueError("No positive edges in train split. Something is wrong with labels or split.")

n_neg = min(len(train_neg), n_pos * neg_mult)

rng = np.random.default_rng(42)
train_neg_sample = rng.choice(train_neg, size=n_neg, replace=False)

train_sel = np.concatenate([train_pos, train_neg_sample])
rng.shuffle(train_sel)

print("CatBoost train_sel:", len(train_sel), "pos:", n_pos, "neg:", n_neg, "pos_rate:", n_pos / len(train_sel))

CatBoost train_sel: 29040 pos: 2640 neg: 26400 pos_rate: 0.09090909090909091


In [21]:
X_train, y_train = build_catboost_Xy(train_sel, edge_index, data.edge_attr, y_edge_t, H)
X_test, y_test = build_catboost_Xy(test_eids, edge_index, data.edge_attr, y_edge_t, H)

print("X_train:", X_train.shape, "X_test:", X_test.shape)

X_train: (29040, 524) X_test: (1384809, 524)


In [22]:
cb = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200,
    task_type="GPU" if False else "CPU"
)

cb.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	test: 0.8727465	best: 0.8727465 (0)	total: 249ms	remaining: 8m 17s
200:	test: 0.9510606	best: 0.9512235 (185)	total: 33s	remaining: 4m 54s
400:	test: 0.9525154	best: 0.9526223 (391)	total: 1m 4s	remaining: 4m 19s
600:	test: 0.9529456	best: 0.9529985 (527)	total: 1m 36s	remaining: 3m 45s
800:	test: 0.9526916	best: 0.9530243 (614)	total: 2m 8s	remaining: 3m 12s
1000:	test: 0.9533386	best: 0.9533386 (1000)	total: 2m 40s	remaining: 2m 40s
1200:	test: 0.9536028	best: 0.9538292 (1168)	total: 3m 11s	remaining: 2m 7s
1400:	test: 0.9539573	best: 0.9540679 (1368)	total: 3m 42s	remaining: 1m 34s
1600:	test: 0.9541776	best: 0.9542136 (1592)	total: 4m 12s	remaining: 1m 2s
1800:	test: 0.9541496	best: 0.9542136 (1592)	total: 4m 42s	remaining: 31.2s
1999:	test: 0.9541592	best: 0.9542312 (1853)	total: 5m 12s	remaining: 0us

bestTest = 0.9542312174
bestIteration = 1853

Shrink model to first 1854 iterations.


In [23]:
# Evaluate CatBoost

p_test = cb.predict_proba(X_test)[:, 1]

print()
print("Pos rate:", float(y_test.mean()))
print("PR-AUC:", average_precision_score(y_test, p_test))
print("ROC-AUC:", roc_auc_score(y_test, p_test))


Pos rate: 0.0006679621521812755
PR-AUC: 0.3054956061779301
ROC-AUC: 0.9542312173954277
